# 29 - Embedding cosine vs ECFP4 Tanimoto, stratified by pocket-embedding similarity

Notebook 24 reports Spearman(ligand-embedding cosine, ECFP4 Tanimoto)
pooled across **all** unique ligand-code pairs, and the manuscript
reads a moderate rho there as evidence of pocket-aware ligand
encoding - but that analysis never conditions on pocket similarity at
all. This notebook adds the missing test, using **pocket-embedding
cosine** (the model's own pocket representation, already used
elsewhere in this project - notebooks 17/21/22 - for comparing
pockets) as the pocket-similarity measure.

Two simple analyses, both per unique ligand-code pair (same
cross-dataset dedup and exact per-pair means as notebook 24), pooled
scope, distinct-code pairs only:

1. **Stratified trend plot** - the same Tanimoto-vs-ligand-cosine
   trend plot as notebook 24/27 (grey scatter + binned median/IQR),
   recomputed separately within each quartile of mean pocket-
   embedding similarity.
2. **Direct correlation** - pocket-embedding cosine vs ligand-
   embedding cosine, plotted and correlated the same way.

Regenerate via
`scripts/build_embedding_pocket_similarity_stratified_notebook.py`.

## Setup

In [ ]:
import pickle
import time
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

PROJECT = Path.cwd().parent / 'DATA' / 'natural_ligands'
MOLS_DIR = PROJECT / 'RESULTS' / 'mols'
POCKETS_DIR = PROJECT / 'RESULTS' / 'pockets'
CCD_MANIFEST = (PROJECT / 'RESULTS' /
                'components_pub_unimol_manifest.tsv')
OUT_DIR = PROJECT / 'RESULTS' / 'embedding_pocket_similarity_stratified'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ['coach420', 'scpdb', 'pdbbind2020', 'holo4k']
N_FOLDS = 6
EMB_DIM = 128
FP_RADIUS, FP_BITS = 2, 2048   # ECFP4

LABEL_FONTSIZE = 13
TICK_FONTSIZE = 13
print('datasets:', DATASETS)

## Load ligand embeddings + ECFP4 fingerprints + pocket embeddings

Same ligand loader as notebook 24, extended to also keep the `pocket`
id column and join each ligand instance to its pocket's embedding
(from `{dataset}_pocket_reps.pkl` - the same file notebook 22 loads
pocket embeddings from), dropping the handful of instances whose
pocket has no embedding on record.

In [ ]:
ccd = pd.read_csv(CCD_MANIFEST, sep='\t', dtype=str)
code_to_smi = dict(zip(ccd['identifier'], ccd['smiles']))

_fp_cache = {}
def fp_for_code(c):
    if c in _fp_cache:
        return _fp_cache[c]
    smi = code_to_smi.get(c, '')
    m = Chem.MolFromSmiles(smi) if smi else None
    fp = None if m is None else AllChem.GetMorganFingerprintAsBitVect(
        m, FP_RADIUS, FP_BITS)
    _fp_cache[c] = fp
    return fp

def load_pocket_embeddings(ds):
    '''pocket id (str) -> (6, 128) embedding array, for one dataset.'''
    names, arr = pd.read_pickle(POCKETS_DIR / f'{ds}_pocket_reps.pkl')
    arr = np.asarray(arr, np.float32)   # (6, P, 128)
    return {str(n): arr[:, i, :] for i, n in enumerate(names)}

def load_dataset(ds):
    with h5py.File(MOLS_DIR / f'bound_{ds}.h5', 'r') as f:
        reps = f['mol_reps'][:]                       # (N, 768)
    n = reps.shape[0]
    emb = reps.reshape(n, N_FOLDS, EMB_DIM).astype(np.float32)
    man = pd.read_csv(MOLS_DIR / f'bound_{ds}_manifest.tsv',
                      sep='\t', dtype=str)
    if len(man) != n:
        raise ValueError(f'{ds}: manifest {len(man)} != h5 {n} rows')
    man = man.sort_values(
        'lmdb_key', key=lambda s: s.astype(str)).reset_index(drop=True)
    codes = man['lig_resname'].fillna('').to_numpy()
    pdb_ids = man['pdb_id'].fillna('').to_numpy()
    pockets = man['pocket'].fillna('').to_numpy()
    fps = [fp_for_code(c) for c in codes]
    pocket_emb_map = load_pocket_embeddings(ds)
    keep = np.array([fp is not None and pk in pocket_emb_map
                     for fp, pk in zip(fps, pockets)])
    pocket_emb = np.stack([pocket_emb_map[pk] for pk in pockets[keep]])
    return {'emb': emb[keep], 'codes': codes[keep], 'pdb_id': pdb_ids[keep],
            'fps': [fp for fp, k in zip(fps, keep) if k],
            'pocket_emb': pocket_emb,
            'n_total': n, 'n_kept': int(keep.sum())}

data = {}
rows = []
for ds in DATASETS:
    d = load_dataset(ds)
    data[ds] = d
    rows.append({'dataset': ds, 'n_mols': d['n_total'],
                 'with_fp_and_pocket_emb': d['n_kept'],
                 'coverage': d['n_kept'] / d['n_total'],
                 'unique_codes': len(set(d['codes']))})
cover = pd.DataFrame(rows)
cover['coverage'] = (cover['coverage'] * 100).round(1)
print(cover.to_string(index=False))

## Pool datasets + cross-dataset dedup

Identical to notebook 24, with the pocket-embedding array carried
through the same dedup mask.

In [ ]:
frames = []
for ds in DATASETS:
    d = data[ds]
    frames.append(pd.DataFrame({'dataset': ds, 'pdb_id': d['pdb_id'],
                                'code': d['codes'],
                                'row': np.arange(len(d['codes']))}))
pooled_man = pd.concat(frames, ignore_index=True)

priority = {ds: i for i, ds in enumerate(DATASETS)}
pooled_man['prio'] = pooled_man['dataset'].map(priority)
chosen = pooled_man.groupby(['pdb_id', 'code'])['prio'].transform('min')
keep_mask = (pooled_man['prio'] == chosen).to_numpy()

emb_pooled_all = np.concatenate([data[ds]['emb'] for ds in DATASETS], axis=0)
fps_pooled_all = sum((data[ds]['fps'] for ds in DATASETS), [])
pemb_pooled_all = np.concatenate([data[ds]['pocket_emb'] for ds in DATASETS], axis=0)
codes_pooled_all = pooled_man['code'].to_numpy()

emb_pooled = emb_pooled_all[keep_mask]
codes_pooled = codes_pooled_all[keep_mask]
fps_pooled = [fp for fp, k in zip(fps_pooled_all, keep_mask) if k]
pemb_pooled = pemb_pooled_all[keep_mask]

print(f'pooled: {len(pooled_man):,} rows -> {keep_mask.sum():,} after '
      f'cross-dataset dedup ({(~keep_mask).sum():,} dropped, '
      f'{len(set(codes_pooled)):,} unique codes)')

SCOPES = {ds: {'codes': data[ds]['codes'], 'fps': data[ds]['fps'],
              'emb': data[ds]['emb'], 'pocket_emb': data[ds]['pocket_emb']}
         for ds in DATASETS}
SCOPES['pooled'] = {'codes': codes_pooled, 'fps': fps_pooled,
                    'emb': emb_pooled, 'pocket_emb': pemb_pooled}
print('\nscopes:', {k: len(v['codes']) for k, v in SCOPES.items()})

## Helper functions

- `full_tanimoto_matrix` - exact pairwise ECFP4 Tanimoto over unique
  ligand codes (same as notebook 24).
- `unit_rows` / average-embedding cosine - identical treatment for
  **both** ligand and pocket embeddings: mean over the 6 folds, unit-
  normalise, one matmul for the full instance x instance cosine
  matrix. No USRCAT, no lmdb atom coordinates, no chunking needed -
  pocket similarity here is exactly as cheap to compute as ligand
  cosine already is.
- `unique_pair_table` - the nb24 core, generalised to also aggregate
  mean pocket-embedding cosine per unique ligand-code pair, alongside
  mean ligand cosine.

In [ ]:
def unit_rows(M):
    nrm = np.linalg.norm(M, axis=1, keepdims=True)
    return M / np.clip(nrm, 1e-12, None)

def full_tanimoto_matrix(fps):
    n = len(fps)
    mat = np.ones((n, n), dtype=np.float32)
    for i in range(n - 1):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[i + 1:])
        mat[i, i + 1:] = sims
        mat[i + 1:, i] = sims
    return mat

def factorize_codes(codes):
    code_idx, uniq_codes = pd.factorize(codes, sort=True)
    return code_idx.astype(np.int32), np.asarray(uniq_codes)

def unique_pair_table(code_idx, n_codes, emb, pocket_emb, tan_mat, uniq_codes):
    E = unit_rows(emb.mean(axis=1))
    P = unit_rows(pocket_emb.mean(axis=1))
    cos_full = (E @ E.T).astype(np.float32)
    pcos_full = (P @ P.T).astype(np.float32)
    iu = np.triu_indices(E.shape[0], k=1)
    df = pd.DataFrame({'key': None, 'cos': cos_full[iu], 'pcos': pcos_full[iu]})
    del cos_full, pcos_full
    ci, cj = code_idx[iu[0]], code_idx[iu[1]]
    lo = np.minimum(ci, cj); hi = np.maximum(ci, cj)
    df['key'] = lo.astype(np.int64) * n_codes + hi.astype(np.int64)
    g = df.groupby('key').agg(mean_cos=('cos', 'mean'),
                              mean_pocket_cos=('pcos', 'mean'),
                              count=('cos', 'count'))
    lo_g = (g.index // n_codes).to_numpy()
    hi_g = (g.index % n_codes).to_numpy()
    n_g = g['count'].to_numpy()
    return pd.DataFrame({
        'code_a': uniq_codes[lo_g], 'code_b': uniq_codes[hi_g],
        'is_self': lo_g == hi_g,
        'n_instance_pairs': n_g,
        'mean_cos': g['mean_cos'].to_numpy(),
        'mean_pocket_cos': g['mean_pocket_cos'].to_numpy(),
        'tanimoto': tan_mat[lo_g, hi_g],
    })

## Compute unique-pair tables (per scope, average embedding)

Per scope: the Tanimoto matrix over unique codes, plus the full
ligand-cosine and pocket-cosine instance matrices, joined into one
unique-ligand-code-pair table. Cached to pickle (per scope) so any
later plot-only tweak can be iterated on without recomputing this -
the slow part is this cell, not the plotting below.

In [ ]:
pair_tables = {}
for scope, s in SCOPES.items():
    t0 = time.time()
    code_idx, uniq_codes = factorize_codes(s['codes'])
    n_codes = len(uniq_codes)
    code_to_fp = {}
    for c, fp in zip(s['codes'], s['fps']):
        code_to_fp.setdefault(c, fp)
    tan_mat = full_tanimoto_matrix([code_to_fp[c] for c in uniq_codes])
    tab = unique_pair_table(code_idx, n_codes, s['emb'], s['pocket_emb'],
                            tan_mat, uniq_codes)
    pair_tables[scope] = tab
    print(f'{scope:12s} n_instances={len(s["codes"]):5d} n_codes={n_codes:5d} '
          f'unique_pairs={len(tab):,}  ({time.time()-t0:.1f}s)')

with open(OUT_DIR / 'pair_tables.pkl', 'wb') as f:
    pickle.dump(pair_tables, f)
print('wrote', OUT_DIR / 'pair_tables.pkl', '(cache for fast plot iteration)')

## Summary: baseline and direct pocket-cosine / ligand-cosine correlation

Per scope, distinct-code pairs only: the notebook-24 baseline
(Spearman(Tanimoto, ligand cosine)) alongside the direct correlation
Spearman(pocket cosine, ligand cosine) - no partial correlation, just
the two plain numbers side by side.

In [ ]:
summary_rows = []
for scope, tab in pair_tables.items():
    dist = tab[~tab['is_self']]
    summary_rows.append({
        'scope': scope, 'n_distinct_pairs': len(dist),
        'rho_tanimoto_cos': spearmanr(dist['tanimoto'], dist['mean_cos']).statistic,
        'rho_pocketcos_cos': spearmanr(dist['mean_pocket_cos'], dist['mean_cos']).statistic,
    })
summary = pd.DataFrame(summary_rows).set_index('scope')
summary.to_csv(OUT_DIR / 'summary_correlations.tsv', sep='\t')
print(summary.round(3).to_string())

## Distribution of pocket-embedding cosine

Pooled scope, distinct-code pairs only. Most mass sits between -0.3
and +0.3, centered near 0, with a long thin tail toward 1.0 - very
few ligand pairs are bound to near-identical or near-opposite
pockets. This shapes which binning schemes below are usable: fixed-
width bins get sparse fast in the tails, while quantile-based bins
(quartiles/deciles) compress that tail into one wide top bin instead.

In [ ]:
dist_all = pair_tables['pooled']
dist_all = dist_all[~dist_all['is_self']]
pc_all = dist_all['mean_pocket_cos']
fig, ax = plt.subplots(figsize=(7, 7))
weights = np.full(len(pc_all), 100.0 / len(pc_all))
ax.hist(pc_all, bins=100, color='0.5', weights=weights)
ax.set_xlabel('mean pocket-embedding cosine similarity', fontsize=LABEL_FONTSIZE)
ax.set_ylabel('% of unique ligand-code pairs', fontsize=LABEL_FONTSIZE)
ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)
ax.set_box_aspect(1)
ax.set_title('Pooled distinct-code pairs: distribution of pocket-embedding cosine')
fig.tight_layout()
out = OUT_DIR / 'fig_pooled_pocket_cosine_distribution.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print('wrote', out)
plt.show()

## 1. Tanimoto-vs-ligand-cosine trend, stratified by pocket-embedding cosine

Pooled scope, distinct-code pairs only. Same trend-plot formatting as
notebook 24/27 (grey background scatter, binned median + IQR), one
panel per bin of `mean_pocket_cos`, so the same relationship notebook
24 reports pooled can be read off separately for ligand pairs bound
to dissimilar vs similar pockets. Three binning schemes, all over the
same data: **quartiles** and **deciles** (equal-N quantile bins, via
`pd.qcut`) and **fixed-width 0.25** bins (`pd.cut`) - chosen over a
finer fixed width since 0.25 keeps every bin in the thousands-to-
millions range (the one exception being the final, near-empty
`> 1.0` sliver, a floating-point artefact, not a real bin).

In [ ]:
def bin_pocket_cos(dist, scheme):
    pc = dist['mean_pocket_cos']
    if scheme == 'quartile':
        return pd.qcut(pc, 4, duplicates='drop')
    if scheme == 'decile':
        return pd.qcut(pc, 10, duplicates='drop')
    if scheme == 'fixed025':
        width = 0.25
        edges = np.arange(np.floor(pc.min() / width) * width,
                          pc.max() + width, width)
        return pd.cut(pc, edges)
    raise ValueError(scheme)

def fmt_interval(lvl):
    # pd.cut/qcut intervals are right-closed, left-open by default -
    # format with the matching bracket so adjacent bins don't look
    # like they overlap at the shared edge.
    lb = '[' if lvl.closed_left else '('
    rb = ']' if lvl.closed_right else ')'
    return f'{lb}{lvl.left:.2f}, {lvl.right:.2f}{rb}'

def draw_trend_panel(ax, sub, color, n_bins=40):
    edges = np.linspace(0, 1, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    bin_idx = np.clip(np.digitize(sub['tanimoto'], edges) - 1, 0, n_bins - 1)
    stats = sub['mean_cos'].groupby(bin_idx).agg(
        median='median', q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75))
    stats = stats.reindex(range(n_bins))
    sizes = 3 + 5 * np.log10(sub['n_instance_pairs'])
    ax.scatter(sub['tanimoto'], sub['mean_cos'], s=sizes, color='0.6',
              alpha=0.05, edgecolors='none', rasterized=True, zorder=1)
    ax.plot(centers, stats['median'], color=color, marker='o', ms=4, zorder=3)
    ax.fill_between(centers, stats['q25'], stats['q75'], color=color,
                    alpha=0.3, zorder=2)
    rho = spearmanr(sub['tanimoto'], sub['mean_cos']).statistic
    ax.set_xlim(0, 1); ax.set_ylim(-1, 1)
    ax.set_xticks(np.arange(0, 1.01, 0.1))
    ax.set_yticks(np.arange(-1, 1.01, 0.25))
    ax.set_box_aspect(1)
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)
    ax.text(0.97, 0.05, r'$N$' + f' = {len(sub):,}\n' + r'$\rho$' +
           f' = {rho:.2f}', transform=ax.transAxes, ha='right', va='bottom',
           fontsize=10,
           bbox=dict(boxstyle='round', facecolor='white', edgecolor='0.5',
                    alpha=0.9))
    return rho

def plot_pocket_binned_trend(scope, scheme, nrows, ncols, scheme_title,
                             out_suffix):
    tab = pair_tables[scope]
    dist = tab[~tab['is_self']].copy()
    dist['bin'] = bin_pocket_cos(dist, scheme)
    levels = [c for c in dist['bin'].cat.categories
             if (dist['bin'] == c).sum() > 0]
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(levels)))

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.6 * ncols, 4.8 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, (lvl, ax) in enumerate(zip(levels, axes)):
        sub = dist[dist['bin'] == lvl]
        draw_trend_panel(ax, sub, colors[i])
        if i % ncols == 0:
            ax.set_ylabel('Ligand embedding cosine similarity',
                         fontsize=LABEL_FONTSIZE)
        ax.set_xlabel('ECFP4 Tanimoto', fontsize=LABEL_FONTSIZE)
        ax.set_title(fmt_interval(lvl), fontsize=11)
    for ax in axes[len(levels):]:
        ax.axis('off')
    fig.suptitle(f'{scope}: Tanimoto vs ligand embedding cosine, by '
                f'pocket embedding cosine ({scheme_title})', fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    out = OUT_DIR / f'fig_{scope}_pocket_{out_suffix}_trend.png'
    fig.savefig(out, dpi=200, bbox_inches='tight')
    print('wrote', out)
    plt.show()

    # Also save each panel as its own standalone figure.
    for i, lvl in enumerate(levels):
        sub = dist[dist['bin'] == lvl]
        fig_i, ax_i = plt.subplots(figsize=(7, 7))
        draw_trend_panel(ax_i, sub, colors[i])
        ax_i.set_xlabel('ECFP4 Tanimoto', fontsize=LABEL_FONTSIZE)
        ax_i.set_ylabel('Ligand embedding cosine similarity',
                       fontsize=LABEL_FONTSIZE)
        ax_i.set_title(f'Pocket embedding cosine in {fmt_interval(lvl)}',
                      fontsize=12)
        fig_i.tight_layout()
        out_i = OUT_DIR / f'fig_{scope}_pocket_{out_suffix}_bin{i+1}.png'
        fig_i.savefig(out_i, dpi=300, bbox_inches='tight')
        print('wrote', out_i)
        plt.close(fig_i)

plot_pocket_binned_trend('pooled', 'quartile', 1, 4, 'quartiles', 'quartile')
plot_pocket_binned_trend('pooled', 'fixed025', 2, 4, 'fixed width 0.25',
                         'fixed025')
plot_pocket_binned_trend('pooled', 'decile', 2, 5, 'deciles', 'decile')

## 2. Pocket-embedding cosine vs ligand-embedding cosine, and vs Tanimoto

Pooled scope, distinct-code pairs only. Direct correlation between
pocket-embedding cosine and (a) ligand-embedding cosine (x = pocket
cosine, fixed [-1, 1] axes - both are embedding-cosine quantities, so
the canonical range is the natural one to show), (b) ECFP4 Tanimoto
(x = Tanimoto, y = pocket cosine, both axes clipped to the data's own
value range rather than the canonical [0, 1]/[-1, 1] - this is the
plain bivariate check behind the `rho(tanimoto, pocket_cos) = 0.13`
figure discussed above: pocket similarity and chemical similarity are
largely independent axes).

In [ ]:
def plot_pocket_cos_vs(scope, ycol, ylabel, color, out_suffix, ylim,
                       n_bins=40):
    tab = pair_tables[scope]
    dist = tab[~tab['is_self']]
    edges = np.linspace(dist['mean_pocket_cos'].min(),
                        dist['mean_pocket_cos'].max(), n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    bin_idx = np.clip(np.digitize(dist['mean_pocket_cos'], edges) - 1, 0, n_bins - 1)
    stats = dist[ycol].groupby(bin_idx).agg(
        median='median', q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75))
    stats = stats.reindex(range(n_bins))
    sizes = 3 + 5 * np.log10(dist['n_instance_pairs'])
    rho = spearmanr(dist['mean_pocket_cos'], dist[ycol]).statistic
    n_pts = len(dist)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(dist['mean_pocket_cos'], dist[ycol], s=sizes, color='0.6',
              alpha=0.05, edgecolors='none', rasterized=True, zorder=1)
    ax.plot(centers, stats['median'], color=color, marker='o', ms=4, zorder=3)
    ax.fill_between(centers, stats['q25'], stats['q75'], color=color,
                    alpha=0.35, zorder=2)
    ax.set_xlabel('Pocket embedding cosine similarity', fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_xlim(-1, 1); ax.set_ylim(*ylim)
    ax.set_xticks(np.arange(-1, 1.01, 0.25))
    ax.set_yticks(np.arange(ylim[0], ylim[1] + 1e-9, 0.25))
    ax.set_box_aspect(1)
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)
    ax.text(0.97, 0.05, r'$N$' + f' = {n_pts:,}\n' + r'$\rho$' +
           f' = {rho:.2f}', transform=ax.transAxes, ha='right', va='bottom',
           fontsize=11,
           bbox=dict(boxstyle='round', facecolor='white', edgecolor='0.5',
                    alpha=0.9))
    fig.tight_layout()
    out = OUT_DIR / f'fig_{scope}_{out_suffix}.png'
    fig.savefig(out, dpi=600, bbox_inches='tight')
    print(f'N={n_pts:,}  rho={rho:.3f}  wrote {out}')
    plt.show()

plot_pocket_cos_vs('pooled', 'mean_cos', 'Ligand embedding cosine similarity',
                   '#9FD8A0', 'pocketcos_vs_ligandcos', (-1, 1))

def plot_tanimoto_vs_pocket_cos(scope='pooled', color='#F6A796', n_bins=40):
    tab = pair_tables[scope]
    dist = tab[~tab['is_self']]
    x = dist['tanimoto']; y = dist['mean_pocket_cos']
    edges = np.linspace(0, 1, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    bin_idx = np.clip(np.digitize(x, edges) - 1, 0, n_bins - 1)
    stats = y.groupby(bin_idx).agg(
        median='median', q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75))
    stats = stats.reindex(range(n_bins))
    sizes = 3 + 5 * np.log10(dist['n_instance_pairs'])
    rho = spearmanr(x, y).statistic
    n_pts = len(dist)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(x, y, s=sizes, color='0.6', alpha=0.05, edgecolors='none',
              rasterized=True, zorder=1)
    ax.plot(centers, stats['median'], color=color, marker='o', ms=4, zorder=3)
    ax.fill_between(centers, stats['q25'], stats['q75'], color=color,
                    alpha=0.35, zorder=2)
    ax.set_xlabel('ECFP4 Tanimoto', fontsize=LABEL_FONTSIZE)
    ax.set_ylabel('Pocket embedding cosine similarity', fontsize=LABEL_FONTSIZE)
    ax.set_xlim(0, 1); ax.set_ylim(-1, 1)
    ax.set_xticks(np.arange(0, 1.01, 0.1))
    ax.set_yticks(np.arange(-1, 1.01, 0.25))
    ax.set_box_aspect(1)
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)
    ax.text(0.97, 0.05, r'$N$' + f' = {n_pts:,}\n' + r'$\rho$' +
           f' = {rho:.2f}', transform=ax.transAxes, ha='right', va='bottom',
           fontsize=11,
           bbox=dict(boxstyle='round', facecolor='white', edgecolor='0.5',
                    alpha=0.9))
    fig.tight_layout()
    out = OUT_DIR / f'fig_{scope}_pocketcos_vs_tanimoto.png'
    fig.savefig(out, dpi=600, bbox_inches='tight')
    print(f'N={n_pts:,}  rho={rho:.3f}  wrote {out}')
    plt.show()

plot_tanimoto_vs_pocket_cos('pooled')

## Takeaways

- The quartile panel reads directly off the same Tanimoto-vs-cosine
  relationship notebook 24 reports pooled, split by how similar the
  bound pockets are (by the model's own pocket embedding). If rho
  climbs from the lowest to the highest pocket-similarity quartile,
  that is direct, simple evidence that the chemistry-cosine
  relationship is not pocket-independent.
- The direct pocket-cosine-vs-ligand-cosine plot asks the more basic
  question underneath the manuscript's claim: do ligands bound to
  similar pockets (by the model's own representation) actually get
  more similar ligand embeddings? A positive rho here is the most
  direct evidence for pocket-conditioned ligand encoding this project
  has produced so far.
- No partial correlation or confound control is applied here - these
  are the two plain, direct readouts the reviewer's comment asks for,
  nothing more.